In [1]:
# Configurar el path para importar los módulos
import sys
import os

# Añadir el directorio raíz del proyecto al path
root_dir = os.chdir(os.path.abspath(os.path.join(os.getcwd(), '..')))
if root_dir not in sys.path:
    sys.path.append(root_dir)

In [2]:
# Importar las funciones necesarias
from db.conexion_db import crear_conexion, cerrar_conexion
from db.querys_db import insert_data, consult_data, update_data
import datetime
import polars as pl
import numpy as np

In [3]:
ruta_archivos_base = r'C:\Users\CGM\Projects\Proyecto CGMExPress\data'
formato = ".xlsx"
nombre_archivo_entregas_ewm_cabezera = "MONITOR_CABEZERA"
nombre_archivo_entregas_ewm_detalle = "MONITOR_DETALLE"
nombre_archivo_guias_remision = "guia_remision"
centros = pl.DataFrame({'centro': ['C200', 'C040', 'C080']})

In [4]:
rename_columns_entregas_ewm = {
    "Documento": "id_entrega",
    "Destinatario mcía.": "codigo_destinatario",
    "Descripción destinatario de mercancías": "descripcion",
    "Clase de documento": "clase_documento",
    "Oficina expedición": "oficina_expedicion",
    "Status salida de mercancías": "status_salida_mercancias",
    "Concluido": "concluido",
    "Status de picking": "status_picking",
    "Cantidad pos.": "cantidad_posiciones",
    "Cantidad de unidades de manipulación": "cantidad_unidades_manipuladas",
    "Cantidad de productos": "cantidad_productos",
    "Autor": "autor",
    "Fecha de entrega (planificada)": "Fecha_entrega_planif",
    "Creados el": "created_at_sap",
    "Creados a la(s)": "created_to_the_sap",
}

In [5]:
excel_entregas_ewm_cabezeraC154 = pl.read_excel(os.path.join(ruta_archivos_base,nombre_archivo_entregas_ewm_cabezera+formato),engine="openpyxl")
excel_entregas_ewm_cabezeraC200 = pl.read_excel(os.path.join(ruta_archivos_base,nombre_archivo_entregas_ewm_cabezera+centros["centro"][0]+formato),engine="openpyxl")
excel_entregas_ewm_cabezeraC040 = pl.read_excel(os.path.join(ruta_archivos_base,nombre_archivo_entregas_ewm_cabezera+centros["centro"][1]+formato),engine="openpyxl")
excel_entregas_ewm_cabezeraC080 = pl.read_excel(os.path.join(ruta_archivos_base,nombre_archivo_entregas_ewm_cabezera+centros["centro"][2]+formato),engine="openpyxl")

In [6]:
df_excel_entregas_ewm_cabezera = pl.concat([excel_entregas_ewm_cabezeraC154, excel_entregas_ewm_cabezeraC200, excel_entregas_ewm_cabezeraC040, excel_entregas_ewm_cabezeraC080])

In [7]:
df_excel_entregas_ewm_cabezera_rename = df_excel_entregas_ewm_cabezera.rename(rename_columns_entregas_ewm)

In [ ]:
connection = crear_conexion("db_almacen_distribucion")
destinatario_db = consult_data(connection, "SELECT id_destinatario,codigo_destinatario FROM tbl_destinatarios")

df_destinatario_db = pl.DataFrame(destinatario_db)

Conexion Exitosa
conexion cerrada


In [27]:
df_destinatario_up_to_db = (
    df_excel_entregas_ewm_cabezera_rename
    .select(
        "codigo_destinatario",
        "descripcion"
    )
    .unique()
    .join(
        df_destinatario_db.select(
            pl.col("codigo_destinatario")
              .alias("codigo_destinatario")
        ),
        on="codigo_destinatario",
        how="anti"
    )
)

In [18]:
last_id = (
    df_destinatario_db
    .select(pl.col("id_destinatario").max())
    .item()
)


In [ ]:
df_destinatario_up_to_db = df_destinatario_up_to_db.with_row_index(
    "id_destinatario",
    offset=last_id + 1
)

In [ ]:
df_destinatario_up_to_db = df_destinatario_up_to_db.with_columns(
    pl.col("id_destinatario").cast(pl.Int64)
)

In [ ]:
df_destinatario_db = pl.concat([
    df_destinatario_db,
    df_destinatario_up_to_db.select(
        "id_destinatario",
        "codigo_destinatario"
    )
])

In [36]:
df_excel_entregas_ewm_cabezera_join_db = df_excel_entregas_ewm_cabezera_rename.join(
    df_destinatario_db,how="left",on="codigo_destinatario"
)

In [37]:
df_excel_entregas_ewm_cabezera_join_db

id_entrega,codigo_destinatario,descripcion,clase_documento,oficina_expedicion,status_salida_mercancias,concluido,status_picking,cantidad_posiciones,cantidad_unidades_manipuladas,cantidad_productos,autor,Fecha_entrega_planif,created_at_sap,created_to_the_sap,id_destinatario
str,str,str,str,str,str,str,str,i64,i64,i64,str,datetime[μs],datetime[μs],time,i64
"""72077801""","""C154""","""LURIN""","""OPM""","""UCL-C154""","""Finalizada""","""Finalizada""","""Finalizado""",1,0,1,"""SAPEWMUSER""",2025-12-17 00:00:00,2025-12-17 00:00:00,09:05:41,1
"""72077805""","""C154""","""LURIN""","""OPM""","""UCL-C154""","""Finalizada""","""Finalizada""","""Finalizado""",1,0,1,"""SAPEWMUSER""",2025-12-17 00:00:00,2025-12-17 00:00:00,09:14:27,1
"""72077816""","""C154""","""LURIN""","""OPM""","""UCL-C154""","""Finalizada""","""Finalizada""","""Finalizado""",4,0,4,"""SAPEWMUSER""",2025-12-17 00:00:00,2025-12-17 00:00:00,09:39:31,1
"""72077817""","""C154""","""LURIN""","""OPM""","""UCL-C154""","""Finalizada""","""Finalizada""","""Finalizado""",1,0,1,"""SAPEWMUSER""",2025-12-17 00:00:00,2025-12-17 00:00:00,09:42:05,1
"""72077840""","""C154""","""LURIN""","""OPM""","""UCL-C154""","""Finalizada""","""Finalizada""","""Finalizado""",2,0,2,"""SAPEWMUSER""",2025-12-17 00:00:00,2025-12-17 00:00:00,12:14:53,1
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""72077842""","""C040""","""AREQUIPA""","""OPM""","""UCL-C040""","""Finalizada""","""Finalizada""","""Finalizado""",29,0,29,"""SAPEWMUSER""",2025-12-17 00:00:00,2025-12-17 00:00:00,12:18:48,6
"""72077845""","""C040""","""AREQUIPA""","""OPM""","""UCL-C040""","""Finalizada""","""Finalizada""","""Finalizado""",28,0,28,"""SAPEWMUSER""",2025-12-17 00:00:00,2025-12-17 00:00:00,12:43:16,6
"""72077848""","""C040""","""AREQUIPA""","""OPM""","""UCL-C040""","""Finalizada""","""Finalizada""","""Finalizado""",26,0,26,"""SAPEWMUSER""",2025-12-17 00:00:00,2025-12-17 00:00:00,13:03:57,6
